# 01. Business Understanding

##  1.1. Business Context
This project focuses on **customer churn prediction** for a B2B SaaS company. 

The company operates on a **subscription-based model**, where revenue is generated through recurring monthly payments. Retaining existing customers is therefore critical to long-term revenue growth, as acquiring new customers is significantly more expensive than retaining current ones.

Customer churn is the event where a customer stops using the service represents a direct revenue loss and often signals underlying product, pricing, or support issues.

##  1.2. Business Problem
The business wants to **identify customers at risk of churning** early enough to take preventive actions such as:
- targeted retention offers
- customer success outreach
- onboarding or education interventions

The challenge is to **prioritize intervention**, since retention actions have a cost and cannot be applied to all customers indiscriminately.

##  1.3. Business Objective
The objective of this project is to build a **predictive model** that estimates the **probability of churn for each customer**, allowing the business to:
- rank customers by churn risk
- focus retention efforts on high-risk, high-value customers
- reduce overall churn rate

##  1.4. Decision Context
The model is intended to support the following decision:
> **“Should we intervene with this customer now?”**

This decision is made before churn occurs, using only information available at that time.
The model output will therefore be a probability of churn, not a binary decision.

##  1.5. Target Variable Definition
**Target:** churn

**Type:** binary classification

**Meaning:**
- 1 — the customer churned
- 0 — the customer remained active

The target represents a historical outcome, used for supervised learning.

# 02. Data Understanding

##  2.1. Dataset Overview
**Data source**

The dataset represents historical customer-level data from a B2B SaaS product operating under a subscription-based business model.  
Each record captures customer characteristics, product usage, engagement signals, and service interactions observed prior to churn.

The data is structured for supervised learning, where historical outcomes are used to predict future churn risk.

**Unit of analysis**

Each row represents **one customer account** observed at a specific point in time.

- One row = one customer
- The dataset is **customer-level**, not event-level
- Features describe customer behavior, account configuration, and support interactions accumulated over time

The target variable (`churn`) indicates whether the customer eventually churned after the observation period.

**Dataset shape**

- Number of rows: 6000 customers  
- Number of columns: 17 features (including the target)

The dataset contains a mix of:
- categorical features (e.g. plan, region, industry)
- numeric features (e.g. tenure, logins per week, NPS)
- binary indicators (e.g. automation_used, discount_used, churn)

##  2.2. Load Data

In [4]:
import pandas as pd

In [12]:
df = pd.read_csv("../data/saas_churn.csv")
df.head()

,customer_id,plan,region,industry,tenure_months,team_size,monthly_price,logins_per_week,campaigns_per_month,automation_used,onboarding_completed,integrations_connected,support_tickets_90d,avg_response_hours,nps,failed_payments_6m,discount_used,churn
0,1,Business,EU,SaaS,25,1,94.16,4.58,6,0,1,0,1,13.4,18.0,1,0,0
1,2,Basic,NaN,SaaS,7,19,19.58,1.13,4,0,1,1,0,NaN,63.0,0,0,1
2,3,Business,EU,E-commerce,18,4,105.14,2.48,4,0,1,2,1,6.0,35.0,0,0,0
3,4,Pro,NaN,Fintech,39,3,32.05,4.58,7,1,1,3,0,NaN,6.0,0,0,1
4,5,Basic,NaN,E-commerce,30,15,23.40,5.77,4,0,1,3,2,16.8,NaN,0,1,0


In [13]:
df.tail()

,customer_id,plan,region,industry,tenure_months,team_size,monthly_price,logins_per_week,campaigns_per_month,automation_used,onboarding_completed,integrations_connected,support_tickets_90d,avg_response_hours,nps,failed_payments_6m,discount_used,churn
5995,5996,Basic,LATAM,E-commerce,38,6,27.35,4.92,5,0,1,1,0,NaN,24.0,0,0,0
5996,5997,Pro,NaN,Agency,46,13,49.13,1.17,4,0,1,2,1,7.5,27.0,0,0,0
5997,5998,Business,NaN,Agency,40,1,101.25,0.52,11,0,1,4,2,15.5,33.0,0,0,0
5998,5999,Business,LATAM,Healthcare,23,3,102.24,4.30,5,1,1,2,3,10.9,NaN,2,0,0
5999,6000,Business,EU,SaaS,13,1,114.03,5.60,3,1,0,0,1,14.9,NaN,0,0,0


##  2.3. Feature Overview
All features represent **customer attributes and behaviors observed prior to churn**.

### 2.3.1 Customer & Account Attributes

**customer_id**: Unique identifier of a customer account.

**plan**: Subscription plan type (e.g. Basic, Business, Pro, Enterprise). Reflects product tier, pricing level, and feature availability.

**region**: Geographic region of the customer. May capture regional market differences, pricing sensitivity, or support coverage.

**industry**: Customer’s industry segment (e.g. SaaS, E-commerce, Fintech). Different industries may exhibit distinct usage patterns and churn dynamics.

**team_size**: Number of active users associated with the account. Serves as a proxy for account size and organizational adoption.

**monthly_price**: Monthly subscription price paid by the customer. Represents customer value and potential price sensitivity.

### 2.3.2 Tenure & Engagement Features

**tenure_months**: Number of months the customer has been subscribed. Longer tenure often correlates with lower churn risk.

**logins_per_week**: Average number of logins per week. Captures product usage frequency and engagement intensity.

**campaigns_per_month**: Average number of campaigns created per month. Reflects active product usage and business reliance on the platform.

**automation_used**: Binary indicator of whether automation features are used. Automation usage often signals deeper product adoption.

**onboarding_completed**: Indicates whether the customer completed the onboarding process. Incomplete onboarding may increase early churn risk.

### 2.3.3 Integration & Support Signals

**integrations_connected**: Number of third-party integrations connected to the account. Integrations increase switching costs and may reduce churn.

**support_tickets_90d**: Number of support tickets created in the last 90 days. High support volume may indicate product friction or dissatisfaction.

**avg_response_hours**: Average response time from customer support, measured in hours. Slow response times can negatively impact customer experience.

**nps**: Net Promoter Score. Represents overall customer satisfaction and loyalty.

### 2.3.4 Billing & Commercial Signals

**failed_payments_6m**: Number of failed payment attempts in the last 6 months. May indicate financial instability or billing friction.

**discount_used**: Binary indicator of whether a discount was applied. Discount usage may reflect price sensitivity or retention efforts.

### 2.3.5 Target Variable

**churn**: Binary target variable indicating whether the customer churned.

- `1` — customer churned  
- `0` — customer remained active  

The target represents a **historical outcome** and is used for supervised learning.

### Feature Semantics & Modeling Assumptions

- All features are observed **before churn occurs**
- No post-churn information is included to avoid data leakage
- Features are aggregated at the **customer level**
- The dataset combines behavioral, operational, and commercial signals